In [ ]:
pip install earthengine-api geemap geopandas pandas numpy scipy matplotlib folium branca shapely fiona pyproj rtree rasterio jupyter notebook ipykernel

In [2]:
# =============================================================================
# PART 1
# GOOGLE EARTH ENGINE EXTRACTION
# MONTHLY WATER BALANCE RASTERS
# CHIRPS + ERA5-LAND PET
# ODISHA SPEI-4 KHARIF WORKFLOW
#
# EXPORTS ONLY:
# MARCH -> SEPTEMBER
#
# REQUIRED FOR:
# JUNE -> SEPTEMBER SPEI-4
# =============================================================================

import ee
import geemap
import geopandas as gpd
from pathlib import Path

# =============================================================================
# INITIALIZE EARTH ENGINE
# =============================================================================

ee.Authenticate()

ee.Initialize(
    project='odishaextreme-heat'
)

# =============================================================================
# PATHS
# =============================================================================

BASE_DIR = Path.cwd()

geojson_path = BASE_DIR / "district.geojson"

# =============================================================================
# LOAD AOI
# =============================================================================

gdf = gpd.read_file(
    geojson_path
)

gdf["geometry"] = gdf.geometry.simplify(
    0.001
)

aoi = geemap.gdf_to_ee(gdf)

# =============================================================================
# PARAMETERS
# =============================================================================

START_YEAR = 1991
END_YEAR = 2025

# March -> September only
MONTHS = [3, 4, 5, 6, 7, 8, 9]

SCALE = 5000

OUTPUT_FOLDER = "ODISHA_SPEI4_WB"

# =============================================================================
# DATASETS
# =============================================================================

# -----------------------------------------------------------------------------
# CHIRPS DAILY PRECIPITATION
# units = mm/day
# -----------------------------------------------------------------------------

chirps = ee.ImageCollection(
    "UCSB-CHG/CHIRPS/DAILY"
)

# -----------------------------------------------------------------------------
# ERA5-LAND MONTHLY AGGREGATED
# PET units = meters
# -----------------------------------------------------------------------------

era5 = ee.ImageCollection(
    "ECMWF/ERA5_LAND/MONTHLY_AGGR"
)

# =============================================================================
# MONTHLY PRECIPITATION
# =============================================================================

def monthly_precip(year, month):

    start = ee.Date.fromYMD(
        year,
        month,
        1
    )

    end = start.advance(
        1,
        "month"
    )

    P = (
        chirps
        .filterDate(start, end)
        .sum()
        .rename("P")
    )

    return P

# =============================================================================
# MONTHLY PET
# =============================================================================

def monthly_pet(year, month):

    start = ee.Date.fromYMD(
        year,
        month,
        1
    )

    end = start.advance(
        1,
        "month"
    )

    img = (
        era5
        .filterDate(start, end)
        .first()
    )

    # ERA5-Land potential evaporation
    # values are negative
    # convert m -> mm

    PET = (
        img
        .select("potential_evaporation_sum")
        .multiply(-1)
        .multiply(1000)
        .rename("PET")
    )

    return PET

# =============================================================================
# EXPORT MONTHLY WATER BALANCE
# =============================================================================

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    for month in MONTHS:

        print(f"\nPreparing {year}_{month:02d}")

        # ---------------------------------------------------------------------
        # PRECIPITATION
        # ---------------------------------------------------------------------

        P = monthly_precip(
            year,
            month
        )

        # ---------------------------------------------------------------------
        # PET
        # ---------------------------------------------------------------------

        PET = monthly_pet(
            year,
            month
        )

        # ---------------------------------------------------------------------
        # WATER BALANCE
        # ---------------------------------------------------------------------

        WB = (
            P.subtract(PET)
            .rename("WB")
            .clip(aoi)
        )

        # ---------------------------------------------------------------------
        # EXPORT TO GOOGLE DRIVE
        # ---------------------------------------------------------------------

        task = ee.batch.Export.image.toDrive(

            image=WB,

            description=f"WB_{year}_{month:02d}",

            folder=OUTPUT_FOLDER,

            fileNamePrefix=f"WB_{year}_{month:02d}",

            region=aoi.geometry(),

            scale=SCALE,

            maxPixels=1e13,

            fileFormat="GeoTIFF"
        )

        task.start()

        print(f"Export started: {year}_{month:02d}")

print("\nAll export tasks submitted.")
print("Open Earth Engine Tasks tab and click RUN.")


Preparing 1991_03
Export started: 1991_03

Preparing 1991_04
Export started: 1991_04

Preparing 1991_05
Export started: 1991_05

Preparing 1991_06
Export started: 1991_06

Preparing 1991_07
Export started: 1991_07

Preparing 1991_08
Export started: 1991_08

Preparing 1991_09
Export started: 1991_09

Preparing 1992_03
Export started: 1992_03

Preparing 1992_04
Export started: 1992_04

Preparing 1992_05
Export started: 1992_05

Preparing 1992_06
Export started: 1992_06

Preparing 1992_07
Export started: 1992_07

Preparing 1992_08
Export started: 1992_08

Preparing 1992_09
Export started: 1992_09

Preparing 1993_03
Export started: 1993_03

Preparing 1993_04
Export started: 1993_04

Preparing 1993_05
Export started: 1993_05

Preparing 1993_06
Export started: 1993_06

Preparing 1993_07
Export started: 1993_07

Preparing 1993_08
Export started: 1993_08

Preparing 1993_09
Export started: 1993_09

Preparing 1994_03
Export started: 1994_03

Preparing 1994_04
Export started: 1994_04

Preparing 

In [1]:
# =============================================================================
# GOOGLE EARTH ENGINE EXTRACTION
# CHIRPS + TERRACLIMATE PET
# MONTHLY WATER BALANCE FOR SPEI-4
#
# ODISHA KHARIF DROUGHT WORKFLOW
# BASELINE: 1991-2020
#
# EXPORTS:
# MARCH -> SEPTEMBER
# =============================================================================

import ee
import geemap
import geopandas as gpd
from pathlib import Path

# =============================================================================
# INITIALIZE EARTH ENGINE
# =============================================================================

ee.Authenticate()

ee.Initialize(
    project='odishaextreme-heat'
)

# =============================================================================
# PATHS
# =============================================================================

BASE_DIR = Path.cwd()

geojson_path = BASE_DIR / "district.geojson"

# =============================================================================
# LOAD AOI
# =============================================================================

gdf = gpd.read_file(
    geojson_path
)

gdf["geometry"] = gdf.geometry.simplify(
    0.001
)

aoi = geemap.gdf_to_ee(gdf)

# =============================================================================
# PARAMETERS
# =============================================================================

START_YEAR = 1991
END_YEAR = 2025

# Required for SPEI-4 Kharif
MONTHS = [3, 4, 5, 6, 7, 8, 9]

SCALE = 4000

OUTPUT_FOLDER = "ODISHA_SPEI4_TERRACLIMATE"

# =============================================================================
# DATASETS
# =============================================================================

# -----------------------------------------------------------------------------
# CHIRPS DAILY PRECIPITATION
# units = mm/day
# -----------------------------------------------------------------------------

chirps = ee.ImageCollection(
    "UCSB-CHG/CHIRPS/DAILY"
)

# -----------------------------------------------------------------------------
# TERRACLIMATE
# PET units = mm/month
# -----------------------------------------------------------------------------

terraclimate = ee.ImageCollection(
    "IDAHO_EPSCOR/TERRACLIMATE"
)

# =============================================================================
# MONTHLY PRECIPITATION
# =============================================================================

def monthly_precip(year, month):

    start = ee.Date.fromYMD(
        year,
        month,
        1
    )

    end = start.advance(
        1,
        "month"
    )

    P = (
        chirps
        .filterDate(start, end)
        .sum()
        .rename("P")
    )

    return P

# =============================================================================
# MONTHLY PET
# =============================================================================

def monthly_pet(year, month):

    start = ee.Date.fromYMD(
        year,
        month,
        1
    )

    end = start.advance(
        1,
        "month"
    )

    PET = (

        terraclimate

        .filterDate(start, end)

        .first()

        .select("pet")

        .multiply(0.1)   # TerraClimate scaling factor

        .rename("PET")

    )

    return PET

# =============================================================================
# EXPORT MONTHLY WATER BALANCE
# =============================================================================

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    for month in MONTHS:

        print(f"\nPreparing {year}_{month:02d}")

        # ---------------------------------------------------------------------
        # PRECIPITATION
        # ---------------------------------------------------------------------

        P = monthly_precip(
            year,
            month
        )

        # ---------------------------------------------------------------------
        # PET
        # ---------------------------------------------------------------------

        PET = monthly_pet(
            year,
            month
        )

        # ---------------------------------------------------------------------
        # WATER BALANCE
        # ---------------------------------------------------------------------

        WB = (

            P.subtract(PET)

            .rename("WB")

            .clip(aoi)

        )

        # ---------------------------------------------------------------------
        # DEBUG VALUES
        # ---------------------------------------------------------------------

        stats = WB.reduceRegion(

            reducer=ee.Reducer.minMax(),

            geometry=aoi.geometry(),

            scale=SCALE,

            maxPixels=1e13

        )

        print(
            f"{year}_{month:02d}",
            stats.getInfo()
        )

        # ---------------------------------------------------------------------
        # EXPORT TO GOOGLE DRIVE
        # ---------------------------------------------------------------------

        task = ee.batch.Export.image.toDrive(

            image=WB,

            description=f"WB_{year}_{month:02d}",

            folder=OUTPUT_FOLDER,

            fileNamePrefix=f"WB_{year}_{month:02d}",

            region=aoi.geometry(),

            scale=SCALE,

            maxPixels=1e13,

            fileFormat="GeoTIFF"
        )

        task.start()

        print(f"Export started: {year}_{month:02d}")

print("\nAll export tasks submitted.")
print("Open Earth Engine Tasks tab and click RUN.")


Preparing 1991_03
1991_03 {'WB_max': -92.357666015625, 'WB_min': -173.70000000000002}
Export started: 1991_03

Preparing 1991_04
1991_04 {'WB_max': -67.2818325519562, 'WB_min': -201.4}
Export started: 1991_04

Preparing 1991_05
1991_05 {'WB_max': -93.71368484497071, 'WB_min': -237.62674098014833}
Export started: 1991_05

Preparing 1991_06
1991_06 {'WB_max': 171.0722273826599, 'WB_min': -45.35479960441589}
Export started: 1991_06

Preparing 1991_07
1991_07 {'WB_max': 651.8738096237182, 'WB_min': 104.7688627243042}
Export started: 1991_07

Preparing 1991_08
1991_08 {'WB_max': 507.3445171356201, 'WB_min': 67.65557107925414}
Export started: 1991_08

Preparing 1991_09
1991_09 {'WB_max': 276.5490135192871, 'WB_min': -66.0692039489746}
Export started: 1991_09

Preparing 1992_03
1992_03 {'WB_max': -134.59257526397707, 'WB_min': -182.20000000000002}
Export started: 1992_03

Preparing 1992_04
1992_04 {'WB_max': -108.59209632873535, 'WB_min': -200.335137462616}
Export started: 1992_04

Preparing

EEException: Image.select: Parameter 'input' is required and may not be null.

In [28]:
# =============================================================================
# GEE JJAS WATER BALANCE RASTER EXPORT
# CHIRPS + TERRACLIMATE PET
# TRUE JJAS SEASONAL WATER BALANCE
# PIXELWISE RASTER EXPORT
# 1991–2010
# =============================================================================

import ee
import geemap
import geopandas as gpd
from pathlib import Path

# =============================================================================
# INITIALIZE EARTH ENGINE
# =============================================================================

ee.Authenticate()

ee.Initialize(
    project='odishaextreme-heat'
)

# =============================================================================
# PATHS
# =============================================================================

BASE_DIR = Path.cwd()

geojson_path = BASE_DIR / "district.geojson"

# =============================================================================
# LOAD AOI
# =============================================================================

gdf = gpd.read_file(
    geojson_path
)

# simplify geometry
gdf["geometry"] = gdf.geometry.simplify(0.001)

# convert to EE
aoi = geemap.gdf_to_ee(gdf)

# =============================================================================
# PARAMETERS
# =============================================================================

START_YEAR = 2025
END_YEAR = 2025

SCALE = 4000

# =============================================================================
# DATASETS
# =============================================================================

# -----------------------------------------------------------------------------
# CHIRPS DAILY PRECIPITATION
# -----------------------------------------------------------------------------

chirps = ee.ImageCollection(
    'UCSB-CHG/CHIRPS/DAILY'
)

# -----------------------------------------------------------------------------
# TERRACLIMATE
# -----------------------------------------------------------------------------

terraclimate = ee.ImageCollection(
    'IDAHO_EPSCOR/TERRACLIMATE'
)

# =============================================================================
# MONTHLY PRECIPITATION
# =============================================================================

def monthly_precip(year, month):

    start = ee.Date.fromYMD(
        year,
        month,
        1
    )

    end = start.advance(
        1,
        'month'
    )

    P = (

        chirps

        .filterDate(start, end)

        .sum()

        .rename('P')

    )

    return P

# =============================================================================
# MONTHLY TERRACLIMATE PET
# =============================================================================

def monthly_pet(year, month):

    start = ee.Date.fromYMD(
        year,
        month,
        1
    )

    end = start.advance(
        1,
        'month'
    )

    PET = (

        terraclimate

        .filterDate(start, end)

        .select('pet')

        .sum()

        .multiply(0.1)

        .rename('PET')

    )

    return PET

# =============================================================================
# JJAS WATER BALANCE EXPORT
# =============================================================================

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    print(f"\nPreparing JJAS raster export for {year}")

    # =========================================================================
    # JJAS MONTHS
    # =========================================================================

    seasonal_wb = []

    for month in [6, 7, 8, 9]:

        print(f"Month {month}")

        # ---------------------------------------------------------------------
        # PRECIPITATION
        # ---------------------------------------------------------------------

        P = monthly_precip(
            year,
            month
        )

        # ---------------------------------------------------------------------
        # PET
        # ---------------------------------------------------------------------

        PET = monthly_pet(
            year,
            month
        )

        # ---------------------------------------------------------------------
        # WATER BALANCE
        # ---------------------------------------------------------------------

        WB = (
            P.subtract(PET)
            .rename('WB')
        )

        seasonal_wb.append(WB)

    # =========================================================================
    # TRUE JJAS TOTAL WATER BALANCE
    # =========================================================================

    JJAS_WB = ee.ImageCollection(
        seasonal_wb
    ).sum()

    JJAS_WB = JJAS_WB.rename(
        f'WB_JJAS_{year}'
    )

    # =========================================================================
    # CLIP TO ODISHA
    # =========================================================================

    JJAS_WB = JJAS_WB.clip(
        aoi.geometry()
    )

    # =========================================================================
    # EXPORT RASTER
    # =========================================================================

    task = ee.batch.Export.image.toDrive(

        image=JJAS_WB,

        description=f'WB_JJAS_{year}',

        folder='DRI_JJAS_TERRACLIMATE',

        fileNamePrefix=f'WB_JJAS_{year}',

        region=aoi.geometry(),

        scale=SCALE,

        maxPixels=1e13,

        fileFormat='GeoTIFF'

    )

    task.start()

    print(f"Export started for {year}")

# =============================================================================
# FINISHED
# =============================================================================

print("\nAll JJAS raster exports submitted.")
print("Open Earth Engine Tasks tab and click RUN.")


Preparing JJAS raster export for 2025
Month 6
Month 7
Month 8
Month 9
Export started for 2025

All JJAS raster exports submitted.
Open Earth Engine Tasks tab and click RUN.


In [14]:
print("OUTPUT_FOLDER:", OUTPUT_FOLDER)
print("Files found:", list(OUTPUT_FOLDER.glob("*")))

OUTPUT_FOLDER: /home/root_1/Documents/CDL/repos/dri/modelling/outputs/csv
Files found: [PosixPath('/home/root_1/Documents/CDL/repos/dri/modelling/outputs/csv/WB_DISTRICTWISE_2014.csv'), PosixPath('/home/root_1/Documents/CDL/repos/dri/modelling/outputs/csv/WB_DISTRICTWISE_2022.csv'), PosixPath('/home/root_1/Documents/CDL/repos/dri/modelling/outputs/csv/WB_DISTRICTWISE_2016.csv'), PosixPath('/home/root_1/Documents/CDL/repos/dri/modelling/outputs/csv/WB_DISTRICTWISE_2015.csv'), PosixPath('/home/root_1/Documents/CDL/repos/dri/modelling/outputs/csv/WB_DISTRICTWISE_2005.csv'), PosixPath('/home/root_1/Documents/CDL/repos/dri/modelling/outputs/csv/WB_DISTRICTWISE_2002.csv'), PosixPath('/home/root_1/Documents/CDL/repos/dri/modelling/outputs/csv/WB_DISTRICTWISE_2021.csv'), PosixPath('/home/root_1/Documents/CDL/repos/dri/modelling/outputs/csv/WB_DISTRICTWISE_2017.csv'), PosixPath('/home/root_1/Documents/CDL/repos/dri/modelling/outputs/csv/WB_DISTRICTWISE_2003.csv'), PosixPath('/home/root_1/Docume